# Golden benchmark — dense vs hybrid + RRF

Notebook này chạy lại **24 golden cases** hiện tại và lưu raw evidence trước khi tạo số tổng hợp. Nó không dùng số liệu trong `RESULT.md` cũ.

Thiết kế A/B: cùng corpus, model, prompt, `top_k=5`; chỉ khác `strategy` (`dense` và `hybrid`). PageIndex, HyDE và cross-encoder đều tắt để so sánh công bằng.

Trước khi chạy: kích hoạt `.venv`, điền `OPENAI_API_KEY`, `LLM_PROVIDER=openai`, `LLM_MODEL=gpt-4o-mini`, `EMBEDDING_PROVIDER=openai`, `EMBEDDING_MODEL=text-embedding-3-small` trong `.env`. Bật `RUN_BENCHMARK = True` ở cell chạy benchmark rồi Run All.


In [8]:
from __future__ import annotations
from dotenv import load_dotenv

import json
import os
import sys
from datetime import datetime, timezone
from pathlib import Path
from statistics import mean

ROOT = Path.cwd().resolve()
if not (ROOT / 'src').is_dir():
    ROOT = ROOT.parent
assert (ROOT / 'src').is_dir(), 'Open this notebook with the repository as cwd.'
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

load_dotenv(ROOT / '.env')

DATASET_PATH = ROOT / 'group_project/evaluation/golden_dataset.json'
ARTIFACT_DIR = ROOT / 'group_project/evaluation/artifacts'
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

golden_cases = json.loads(DATASET_PATH.read_text(encoding='utf-8'))
required_fields = {'id', 'question', 'expected_answer',
                   'expected_context', 'answer_sources'}
assert len(
    golden_cases) == 24, f'Expected the reviewed 24-case set, found {len(golden_cases)}'
assert len({case['id'] for case in golden_cases}) == len(golden_cases)
assert all(required_fields <= case.keys() for case in golden_cases)
assert os.getenv(
    'OPENAI_API_KEY'), 'Set OPENAI_API_KEY in .env before running.'
print(f'Loaded {len(golden_cases)} reviewed golden cases from {DATASET_PATH.relative_to(ROOT)}')
print('Generator:', os.getenv('LLM_PROVIDER'), os.getenv('LLM_MODEL'))
print('Embedding:', os.getenv('EMBEDDING_PROVIDER'), os.getenv('EMBEDDING_MODEL'))

Loaded 24 reviewed golden cases from group_project/evaluation/golden_dataset.json
Generator: openai gpt-4o-mini
Embedding: openai text-embedding-3-small


In [9]:
from src.pipeline_config import PipelineConfig
from src.task10_generation import generate_with_config

defaults = PipelineConfig.defaults()
assert defaults.provider == 'openai', 'This notebook currently uses OpenAI for generation and Ragas judging.'
assert defaults.model, 'Set LLM_MODEL in .env.'

COMMON = dict(
    top_k=5,
    score_threshold=defaults.score_threshold,
    rrf_k=60,
    use_pageindex=False,
    use_hyde=False,
    use_model_rerank=False,
    provider=defaults.provider,
    model=defaults.model,
    temperature=0.0,
    response_style='concise',
    show_trace=False,
)
CONFIGURATIONS = {
    'dense': PipelineConfig(strategy='dense', **COMMON),
    'hybrid_rrf': PipelineConfig(strategy='hybrid', **COMMON),
}
CONFIGURATIONS

{'dense': PipelineConfig(strategy='dense', top_k=5, score_threshold=0.3, rrf_k=60, use_pageindex=False, use_hyde=False, use_model_rerank=False, provider='openai', model='gpt-4o-mini', temperature=0.0, response_style='concise', show_trace=False),
 'hybrid_rrf': PipelineConfig(strategy='hybrid', top_k=5, score_threshold=0.3, rrf_k=60, use_pageindex=False, use_hyde=False, use_model_rerank=False, provider='openai', model='gpt-4o-mini', temperature=0.0, response_style='concise', show_trace=False)}

In [10]:
RUN_ID = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')
RAW_PATH = ARTIFACT_DIR / f'golden-benchmark-{RUN_ID}.raw.json'
SCORED_PATH = ARTIFACT_DIR / f'golden-benchmark-{RUN_ID}.scored.json'


def write_json(path: Path, payload: dict) -> None:
    path.write_text(json.dumps(payload, ensure_ascii=False,
                    indent=2), encoding='utf-8')


def source_snapshot(source: dict) -> dict:
    return {
        'id': source['id'],
        'score': source['score'],
        'retrieval_method': source['retrieval_method'],
        'content': source['content'],
        'metadata': source['metadata'],
    }


def empty_raw_artifact() -> dict:
    return {
        'run_id': RUN_ID,
        'started_at': datetime.now(timezone.utc).isoformat(),
        'case_count': len(golden_cases),
        'case_ids': [case['id'] for case in golden_cases],
        'generator_model': defaults.model,
        'embedding_model': os.getenv('EMBEDDING_MODEL'),
        'configurations': {name: config.public_dict() for name, config in CONFIGURATIONS.items()},
        'runs': [],
    }


def run_one_configuration(artifact: dict, name: str, config: PipelineConfig) -> None:
    completed = {(row['configuration'], row['id']) for row in artifact['runs']}
    for index, case in enumerate(golden_cases, 1):
        if (name, case['id']) in completed:
            continue
        print(f'[{name} {index}/{len(golden_cases)}] {case["id"]}', flush=True)
        result = generate_with_config(case['question'], config)
        artifact['runs'].append({
            'configuration': name,
            'id': case['id'],
            'question': case['question'],
            'expected_answer': case['expected_answer'],
            'expected_context': case['expected_context'],
            'answer': result['answer'],
            'retrieval_source': result['retrieval_source'],
            'sources': [source_snapshot(source) for source in result['sources']],
        })
        write_json(RAW_PATH, artifact)  # checkpoint after every paid request

In [11]:
# Deliberately false: opening or running all cells must never spend API budget by accident.
RUN_BENCHMARK = True

if RUN_BENCHMARK:
    artifact = empty_raw_artifact()
    for name, config in CONFIGURATIONS.items():
        run_one_configuration(artifact, name, config)
    artifact['finished_at'] = datetime.now(timezone.utc).isoformat()
    write_json(RAW_PATH, artifact)
    print(f'Raw artifact saved: {RAW_PATH.relative_to(ROOT)}')
else:
    print('Set RUN_BENCHMARK = True only when you are ready to call OpenAI.')

[dense 1/24] GQ-001
[dense 2/24] GQ-002
[dense 3/24] GQ-003
[dense 4/24] GQ-004
[dense 5/24] GQ-005
[dense 6/24] GQ-006
[dense 7/24] GQ-007
[dense 8/24] GQ-008
[dense 9/24] GQ-009
[dense 10/24] GQ-010
[dense 11/24] GQ-011
[dense 12/24] GQ-012
[dense 13/24] GQ-013
[dense 14/24] GQ-014
[dense 15/24] GQ-015
[dense 16/24] GQ-016
[dense 17/24] GQ-017
[dense 18/24] GQ-018
[dense 19/24] GQ-019
[dense 20/24] GQ-020
[dense 21/24] GQ-021
[dense 22/24] GQ-022
[dense 23/24] GQ-023
[dense 24/24] GQ-024
[hybrid_rrf 1/24] GQ-001
[hybrid_rrf 2/24] GQ-002
[hybrid_rrf 3/24] GQ-003
[hybrid_rrf 4/24] GQ-004
[hybrid_rrf 5/24] GQ-005
[hybrid_rrf 6/24] GQ-006
[hybrid_rrf 7/24] GQ-007
[hybrid_rrf 8/24] GQ-008
[hybrid_rrf 9/24] GQ-009
[hybrid_rrf 10/24] GQ-010
[hybrid_rrf 11/24] GQ-011
[hybrid_rrf 12/24] GQ-012
[hybrid_rrf 13/24] GQ-013
[hybrid_rrf 14/24] GQ-014
[hybrid_rrf 15/24] GQ-015
[hybrid_rrf 16/24] GQ-016
[hybrid_rrf 17/24] GQ-017
[hybrid_rrf 18/24] GQ-018
[hybrid_rrf 19/24] GQ-019
[hybrid_rrf 20/24] G

## Ragas scoring

Run this only after raw generation completed. It scores the same stored answer/context, so judging can be resumed without re-generating answers. The output includes all four required metrics and preserves any failed metric as an explicit error instead of inventing a score.


In [12]:
# Requires: pip install -e '.[dev]'  (ragas is already declared in pyproject.toml)
from openai import AsyncOpenAI
from ragas.embeddings.base import embedding_factory
from ragas.llms.base import llm_factory
from ragas.metrics.collections import AnswerRelevancy, ContextPrecision, ContextRecall, Faithfulness

judge_client = AsyncOpenAI(api_key=os.environ['OPENAI_API_KEY'])
judge_llm = llm_factory(defaults.model, provider='openai', client=judge_client)
judge_embeddings = embedding_factory(
    'openai', model=os.getenv('EMBEDDING_MODEL'), client=judge_client)
METRICS = {
    'faithfulness': Faithfulness(llm=judge_llm),
    'answer_relevance': AnswerRelevancy(llm=judge_llm, embeddings=judge_embeddings),
    'context_recall': ContextRecall(llm=judge_llm),
    'context_precision': ContextPrecision(llm=judge_llm),
}


async def score_record(record: dict) -> dict:
    contexts = [source['content'] for source in record['sources']]
    if not contexts:
        return {'error': 'no_retrieved_context'}
    try:
        return {
            'faithfulness': float((await METRICS['faithfulness'].ascore(
                user_input=record['question'], response=record['answer'], retrieved_contexts=contexts)).value),
            'answer_relevance': float((await METRICS['answer_relevance'].ascore(
                user_input=record['question'], response=record['answer'])).value),
            'context_recall': float((await METRICS['context_recall'].ascore(
                user_input=record['question'], retrieved_contexts=contexts, reference=record['expected_answer'])).value),
            'context_precision': float((await METRICS['context_precision'].ascore(
                user_input=record['question'], reference=record['expected_answer'], retrieved_contexts=contexts)).value),
        }
    except Exception as error:
        return {'error': f'{type(error).__name__}: {error}'}

In [15]:
RUN_SCORING = True

if RUN_SCORING:
    raw = json.loads(RAW_PATH.read_text(encoding='utf-8'))
    for index, record in enumerate(raw['runs'], 1):
        if 'scores' in record:
            continue
        print(
            f'[score {index}/{len(raw["runs"])}] {record["configuration"]} {record["id"]}', flush=True)
        record['scores'] = await score_record(record)
        write_json(SCORED_PATH, raw)
    raw['scored_at'] = datetime.now(timezone.utc).isoformat()
    write_json(SCORED_PATH, raw)
    print(f'Scored artifact saved: {SCORED_PATH.relative_to(ROOT)}')
else:
    print('After generation, set RUN_SCORING = True and run this cell.')

[score 1/48] dense GQ-001
[score 2/48] dense GQ-002
[score 3/48] dense GQ-003
[score 4/48] dense GQ-004
[score 5/48] dense GQ-005
[score 6/48] dense GQ-006
[score 7/48] dense GQ-007
[score 8/48] dense GQ-008
[score 9/48] dense GQ-009
[score 10/48] dense GQ-010
[score 11/48] dense GQ-011
[score 12/48] dense GQ-012
[score 13/48] dense GQ-013
[score 14/48] dense GQ-014
[score 15/48] dense GQ-015
[score 16/48] dense GQ-016
[score 17/48] dense GQ-017
[score 18/48] dense GQ-018
[score 19/48] dense GQ-019
[score 20/48] dense GQ-020
[score 21/48] dense GQ-021
[score 22/48] dense GQ-022
[score 23/48] dense GQ-023
[score 24/48] dense GQ-024
[score 25/48] hybrid_rrf GQ-001
[score 26/48] hybrid_rrf GQ-002
[score 27/48] hybrid_rrf GQ-003
[score 28/48] hybrid_rrf GQ-004
[score 29/48] hybrid_rrf GQ-005
[score 30/48] hybrid_rrf GQ-006
[score 31/48] hybrid_rrf GQ-007
[score 32/48] hybrid_rrf GQ-008
[score 33/48] hybrid_rrf GQ-009
[score 34/48] hybrid_rrf GQ-010
[score 35/48] hybrid_rrf GQ-011
[score 36

In [16]:
# Run after scoring. This creates a reviewable summary, not a fabricated RESULT.md overwrite.
if SCORED_PATH.exists():
    scored = json.loads(SCORED_PATH.read_text(encoding='utf-8'))
    metric_names = ('faithfulness', 'answer_relevance',
                    'context_recall', 'context_precision')
    summary = {}
    for configuration in CONFIGURATIONS:
        rows = [row for row in scored['runs'] if row['configuration'] ==
                configuration and set(metric_names) <= set(row.get('scores', {}))]
        summary[configuration] = {metric: mean(
            row['scores'][metric] for row in rows) for metric in metric_names}
        summary[configuration]['scored_cases'] = len(rows)
    print(json.dumps(summary, ensure_ascii=False, indent=2))
    assert all(item['scored_cases'] == 24 for item in summary.values()
               ), 'Do not fill RESULT.md until all 24 cases score.'
else:
    print('No scored artifact yet.')

{
  "dense": {
    "faithfulness": 0.8090277777777778,
    "answer_relevance": 0.49992863533052007,
    "context_recall": 0.7847222222222222,
    "context_precision": 0.8390046296047386,
    "scored_cases": 24
  },
  "hybrid_rrf": {
    "faithfulness": 0.8229166666666666,
    "answer_relevance": 0.4896172281099882,
    "context_recall": 0.8263888888888888,
    "context_precision": 0.9041666666427662,
    "scored_cases": 24
  }
}
